# Week 08: Building the AI/ML pipeline

Last week you built every diffusion-specific piece of mathematics the model
needs — in pure numpy, with every line transparent and debuggable. The
**forward process** that maps a clean residual to noise across T = 200 timesteps,
the **cosine noise schedule** that defines α_t and σ_t at every t, and the
empirical verifications that the corruption behaves the way the math says it
should. The window table, the stratified split, and the schedule are all
saved to `diffusion_windows.parquet`. What you have *not* built is anything
with learnable parameters.

That changes this week. The forward apparatus from Week 07 will be reused
unchanged; on top of it we add the **reverse process** — a small denoising
neural network whose job is to estimate ε from r_t, trained in PyTorch with
PyTorch Lightning. By the end of this notebook you will have an end-to-end
**unconditional diffusion model** that learns the marginal distribution of
residuals, samples from it, and — through a deliberate architectural
ablation — answers a real empirical question: does the network's awareness
of the noise level (the **timestep embedding**) actually earn its keep on
this dataset?

**Strategic context.** This week is the most code-heavy of the three. The
conceptual lift is smaller than Week 07's diffusion math — the hard ideas
(forward process, schedule, ε-prediction) are already in your toolkit. What
this week demands is *integration*: turning the numpy schedule into a
PyTorch buffer, turning the residual table into a Dataset, turning the
training equation into a Lightning training step, and watching the whole
thing learn. Most of you will encounter the canonical AI/ML pipeline
(Dataset → DataLoader → Model → LightningModule → Trainer) for the first
time here. The pipeline is not diffusion-specific — every PyTorch project
you ever build will have the same five pieces — but each piece has a
diffusion-specific wrinkle, and we will flag those wrinkles as we go.

**One deliberate omission this week: conditioning.** The Week 09 model will
condition on (cycle amplitude, universal-path latitude) and run through
`compute_global_nll`. This week we train the *unconditional* version: it
learns the marginal distribution of residuals across all training cycles
and can sample plausible residuals, but it cannot target a specific window.
The reason for the split is pedagogical: build the diffusion machinery
first, feel it work end-to-end, and only then add the conditioning layer
on top. One new idea per week.

**By the end of this notebook you should be able to:**
- Wrap the Week 07 residual table (loaded from `diffusion_windows.parquet`)
  in a **PyTorch Dataset** and explain why the dataset for diffusion training
  returns clean residuals only — without any noise injection or timestep
  sampling.
- Build the three **DataLoaders** (train / val / test) using the `split`
  column already present in the parquet, and articulate why shuffling
  matters for training but not for validation.
- Implement a **sinusoidal timestep embedding** that maps the integer t to a
  dense vector, and visualize how the embedding rotates through embedding
  space as t varies.
- Build a **DiffusionMLP** that takes (r_t, t) and returns ε̂, with a
  constructor flag that toggles whether the timestep embedding is used at
  all — the architectural ablation that lets you measure whether t-awareness
  earns its keep.
- Write a **PyTorch Lightning training loop** that samples t per batch
  element, draws ε, applies the Week 07 forward equation in PyTorch, and
  trains on MSE loss against the true ε.
- Implement a **DDIM-style sampler** that runs the trained model in reverse
  to generate new residuals, and verify that the sampled residuals match
  the training distribution on bin-wise mean, standard deviation, and
  covariance structure.
- Run the with-vs-without-timestep-embedding ablation and read its result
  honestly, including the case where it tells you something uncomfortable
  about your architecture choice.


In [ ]:
import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    subprocess.run(["git", "-C", repo_path, "pull"], check=True)
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

In [ ]:
# ── locate repo root robustly (same scheme as Week 07) ──────────────────────
_cwd = os.getcwd()
_week8_dir = _repo_root = None
for _base in [_cwd] + [os.path.abspath(os.path.join(_cwd, *[".."] * i)) for i in range(1, 5)]:
    if os.path.isfile(os.path.join(_base, "weeks", "week_08", "butterflAI_model.py")):
        _week8_dir = os.path.join(_base, "weeks", "week_08")
        _repo_root = _base
        break
    if os.path.isfile(os.path.join(_base, "butterflAI_model.py")) and "week_08" in _base:
        _week8_dir = _base
        _repo_root = os.path.abspath(os.path.join(_base, "../.."))
        break

if _week8_dir is None:
    raise FileNotFoundError(
        "Cannot locate butterflAI_model.py. Make sure repo_path points to the "
        "butterflai repo root, and that you have run Week 07 to produce "
        "diffusion_windows.parquet."
    )
for _p in [_week8_dir, _repo_root]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ── load Week 07 outputs ─────────────────────────────────────────────────────
windows_df = pd.read_parquet(Path(_week8_dir) / "diffusion_windows.parquet")

hist_cols = [f"hist_emp_{j:02d}" for j in range(15)]
par_cols  = [f"hist_par_{j:02d}" for j in range(15)]

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

# ── load the official ButterflAI classical model ─────────────────────────────
from butterflAI_model import ButterflAIModel
classical = ButterflAIModel(os.path.join(_week8_dir, "official_model.npz"))

print(f"Loaded {len(windows_df)} windows from diffusion_windows.parquet")
print(f"  splits : {windows_df['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles : {sorted(windows_df['cycle'].unique())}")
print(classical)

In [ ]:
# ── recompute the Week 07 cosine schedule (numpy version) ──────────────────
# Same formula as Week 07 Task 28. Kept here so this notebook is
# self-contained — the LightningModule in Task 40 will build a torch
# version of the same schedule from these arrays.
T, s_offset = 200, 0.008
_t_arr         = np.arange(T + 1, dtype=float)
_f0            = np.cos(np.pi / 2 * s_offset / (1 + s_offset)) ** 2
_alpha_bar_raw = np.cos(np.pi / 2 * (_t_arr / T + s_offset) / (1 + s_offset)) ** 2 / _f0
_alpha_bar_raw = np.clip(_alpha_bar_raw, 0.0, 1.0)
_beta = np.zeros(T + 1)
for t in range(1, T + 1):
    _beta[t] = np.clip(1.0 - _alpha_bar_raw[t] / _alpha_bar_raw[t - 1], 1e-8, 0.999)
alpha_bar_np = np.ones(T + 1)
for t in range(1, T + 1):
    alpha_bar_np[t] = alpha_bar_np[t - 1] * (1.0 - _beta[t])
alpha_np = np.sqrt(alpha_bar_np)
sigma_np = np.sqrt(np.clip(1.0 - alpha_bar_np, 0.0, 1.0))

print(f"T={T}, schedule arrays length {len(alpha_np)}")

In [ ]:
# ── Inspect the classical model on any cycle in windows_df ─────────────────
# Change cycle_number to see a different cycle.  Both hemispheres are shown
# simultaneously.  Filled profiles = empirical 6-month histogram (hist_emp_*);
# dashed curves = classical ButterflAI Gaussian evaluated at the same (A, τ).
cycle_number  = 24      # ← any value in sorted(windows_df['cycle'].unique())
PROFILE_SCALE = 0.40    # max density → this many τ-years of horizontal width

# ── select & sort windows ─────────────────────────────────────────────────
wdf_n = (windows_df[(windows_df["cycle"] == cycle_number) &
                     (windows_df["hemisphere"] == "north")]
         .sort_values("tau_center").reset_index(drop=True))
wdf_s = (windows_df[(windows_df["cycle"] == cycle_number) &
                     (windows_df["hemisphere"] == "south")]
         .sort_values("tau_center").reset_index(drop=True))

if wdf_n.empty and wdf_s.empty:
    raise ValueError(
        f"Cycle {cycle_number} not found. "
        f"Available: {sorted(windows_df['cycle'].unique())}"
    )

# Shared normalization across both hemispheres so relative amplitudes are comparable
_cyc_mask  = windows_df["cycle"] == cycle_number
global_max = max(windows_df[_cyc_mask][hist_cols].values.max(), 1e-9)
scale      = PROFILE_SCALE / global_max

# Shared τ range for colormap — same colour = same time in both panels
_all_tau   = pd.concat([wdf_n["tau_center"], wdf_s["tau_center"]])
tau_norm   = plt.Normalize(vmin=_all_tau.min(), vmax=_all_tau.max())
cmap_win   = plt.get_cmap("viridis")

# Step-function bin edges for fill_betweenx  (15 bins → 30 edge positions)
_lat_edges_step = np.concatenate(
    [[LAT_BINS[0]], np.repeat(LAT_BINS[1:-1], 2), [LAT_BINS[-1]]]
)   # shape (30,)
_lat_fine = np.linspace(0, 45, 300)   # fine grid for smooth Gaussian curves

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.subplots_adjust(hspace=0.06)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

for ax, wdf, sign, hemi_label in [
    (axes[0], wdf_n, +1, "North"),
    (axes[1], wdf_s, -1, "South"),
]:
    if wdf.empty:
        ax.text(0.5, 0.5, f"No data for {hemi_label}",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=13, color="gray")
        ax.set_ylabel("Latitude (°)")
        continue

    for _, row in wdf.iterrows():
        tau   = row["tau_center"]
        A     = row["amplitude"]
        color = cmap_win(tau_norm(tau))

        # ── empirical histogram: filled step-function profile ────────────
        hist_vals = row[hist_cols].values.astype(float)
        hist_step = np.repeat(hist_vals, 2)        # (30,)
        x_hist    = tau + hist_step * scale

        lats_plot = sign * _lat_edges_step
        ax.fill_betweenx(lats_plot, tau, x_hist,
                         color=color, alpha=0.35, linewidth=0, zorder=2)
        ax.plot(x_hist, lats_plot,
                color=color, linewidth=0.8, alpha=0.75, zorder=3)

        # ── classical Gaussian: smooth dashed curve ───────────────────────
        gauss_vals = classical.density(A, tau, _lat_fine)
        x_gauss    = tau + gauss_vals * scale
        ax.plot(x_gauss, sign * _lat_fine,
                color=color, linewidth=2.0, linestyle="--", alpha=0.9, zorder=5)

    ax.set_ylabel(f"Latitude (°) — {hemi_label}", fontsize=10)
    ax.set_ylim((0, 48) if sign == 1 else (-48, 0))

    if sign == 1:   # legend once, in the north panel
        ax.legend(
            handles=[
                Patch(facecolor="gray", alpha=0.45,
                      label="Empirical histogram (hist_emp_*)"),
                Line2D([0], [0], color="gray", linewidth=2, linestyle="--",
                       label="Classical Gaussian — ButterflAI model"),
            ],
            loc="upper right", fontsize=9,
        )

# ── single shared colorbar spanning both panels ───────────────────────────
sm = plt.cm.ScalarMappable(cmap="viridis", norm=tau_norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=axes, pad=0.01, fraction=0.02)
cb.set_label("τ (yr)", fontsize=10)

axes[1].set_xlabel("τ  (years from reference epoch)", fontsize=11)
fig.suptitle(
    f"Cycle {cycle_number}  —  6-month windows: "
    "empirical histograms (filled) overplotted with classical ButterflAI Gaussians (dashed)\n"
    "Profiles share the same horizontal scale; colour = τ (same scale in both panels)",
    fontsize=11,
)
plt.show()

---
# Week 08 Tasks: Building the AI/ML pipeline

Everything above this line is **setup**: the Week 07 outputs are loaded,
the cosine schedule is recomputed in numpy for reference, and the
official `ButterflAIModel` is instantiated. Below this line begins the
Week 08 work proper: build the dataset, the model, the training loop, the
sampler, and the ablation comparison.

The tasks build sequentially. Tasks 33–35 wrap the Week 07 residual table
in a PyTorch Dataset and DataLoader. Tasks 36–39 build the model in three
layered pieces (sinusoidal embedding → embedding module → main network)
and then sanity-test it. Task 40 stitches the model and the schedule into
a LightningModule. Task 41 trains it. Tasks 42–43 implement sampling and
verify the sampled distribution matches the training distribution. Task 44
runs the ablation experiment and produces the comparison table that closes
the week.

A note on convention: throughout this notebook, all tensor shape
manipulation uses `einops.rearrange` and `einops.repeat` with named
dimensions rather than `view`, `reshape`, `squeeze`, `unsqueeze`,
`expand`, or `tile`. The named-dimension form makes the shape semantics
explicit, which is especially useful when multiple dimensions could
plausibly be the batch dimension. `einops` was already required for
Week 07 and is available in the environment.


---
## Task 33 — Wrap the residual table in a PyTorch Dataset

The Week 07 residual table has one row per (cycle, hemisphere, 6-month
window) and a `split` column tagging each row as `'train'`, `'val'`, or
`'test'`. Most diffusion training pipelines call this object a **Dataset**:
an indexed collection of clean data points that the training loop will
draw from at random.

The dataset for
diffusion training is simple. In supervised learning  `__getitem__` typically returns an (input, target) pair. For diffusion, that is
not the right factoring. The triple (r_t, t, ε) the network actually trains
on is generated *inside* the Lightning training step, not in `__getitem__`.
The dataset returns just the clean residual r; the timestep t is sampled
fresh per batch, the noise ε is drawn fresh per batch, and the corruption
r_t = α_t · r + σ_t · ε is computed on the fly using the same formula
your Week 07 `forward_corrupt` implemented. This separation is deliberate:
it lets you change the timestep sampling distribution, the ε distribution,
or the loss weighting without ever touching the dataset.

This week the dataset returns *only*
the residual, as a tensor of shape (15,). Week 09 will add conditioning on
(amplitude, mu_universal), which means the dataset will need to return
those covariates alongside the residual. We do not preemptively wire that
in this week — keeping the dataset minimal makes the diffusion-specific
machinery easier to see — but the parquet already contains those columns,
so the Week 09 refactor will be small.

**Tasks:**
- Define a class `ResidualDataset(torch.utils.data.Dataset)` that wraps
  `windows_df`.
- The constructor takes the full `windows_df` and a string `split` in
  `{'train', 'val', 'test'}`, and filters the DataFrame to retain only
  rows whose `split` column matches.
- `__len__` returns the number of rows after filtering.
- `__getitem__(idx)` extracts the residual (the difference between the 15
  `hist_emp_*` columns and the 15 `hist_par_*` columns) and returns it as
  a single `torch.float32` tensor of shape `(15,)`.

**Important:** the residual is the *difference* between the empirical and
parametric histograms (both densities). It is not normalized to unit
variance, not centered at zero, and not bounded — it is in the same units
as the histograms (probability density per latitude). The diffusion model
will learn to generate residuals with whatever distributional structure
the training data exhibits, so do not preprocess them here. The
`hist_par_*` columns themselves were generated in Week 07 by integrating
the official classical model's per-window Gaussian over each bin; the
residual you return here is the part of the per-window density the
classical model leaves on the table.


In [ ]:
# Put your code here for Task 33.
# Task 33: ResidualDataset
# Depends on: windows_df, emp_cols, par_cols (from setup)

import torch
from torch.utils.data import Dataset

class ResidualDataset(Dataset):
    """
    Wraps the Week 07 window table for diffusion training.

    Returns one clean residual tensor per index — no noise injection,
    no timestep sampling. Both happen inside the Lightning training step
    so that every draw of a given residual sees a fresh (t, ε) pair.

    Parameters
    ----------
    windows_df : pd.DataFrame
        Full window table from Task 26, with 'split' column.
    split : str
        One of 'train', 'val', 'test'. Only rows matching this tag
        are retained.

    __getitem__ returns
    -------------------
    r : torch.float32 tensor, shape (15,)
        hist_emp − hist_par for that window, in density units.
        Not normalized, not centered — raw residual as computed in Task 26.
    """

    # Column lists are class-level constants so they're computed once
    _EMP_COLS = [f"hist_emp_{k:02d}" for k in range(15)]
    _PAR_COLS = [f"hist_par_{k:02d}" for k in range(15)]

    def __init__(self, windows_df: pd.DataFrame, split: str):
        assert split in {"train", "val", "test"}, \
            f"split must be 'train', 'val', or 'test', got '{split}'"

        # Filter to the requested split
        mask = windows_df["split"] == split
        self._df = windows_df[mask].reset_index(drop=True)

        # Pre-compute residuals as a numpy array once at construction time.
        # Shape: (N, 15).  Avoids repeated pandas column lookups in __getitem__.
        emp = self._df[self._EMP_COLS].values.astype(np.float32)
        par = self._df[self._PAR_COLS].values.astype(np.float32)
        self._residuals = emp - par   # shape (N, 15)

        # Keep metadata for inspection / debugging
        self.split      = split
        self.n_windows  = len(self._df)
        self.n_bins     = 15

        print(f"ResidualDataset(split='{split}'): {self.n_windows} windows  "
              f"| residual shape per item: ({self.n_bins},)")
        print(f"  residual range: [{self._residuals.min():.4f}, "
              f"{self._residuals.max():.4f}]")
        print(f"  residual mean : {self._residuals.mean():.5f}  "
              f"std: {self._residuals.std():.5f}")

    def __len__(self) -> int:
        return self.n_windows

    def __getitem__(self, idx: int) -> torch.Tensor:
        # Convert the pre-computed numpy row to a float32 tensor
        # shape: (15,)
        return torch.from_numpy(self._residuals[idx])

    # ── Convenience methods ───────────────────────────────────────────────
    def all_residuals(self) -> np.ndarray:
        """Return all residuals as a numpy array (N, 15) — useful for stats."""
        return self._residuals.copy()

    def metadata(self, idx: int) -> pd.Series:
        """Return the full metadata row for a given index."""
        return self._df.iloc[idx]


# ── Instantiate all three splits ──────────────────────────────────────────
ds_train = ResidualDataset(windows_df, "train")
ds_val   = ResidualDataset(windows_df, "val")
ds_test  = ResidualDataset(windows_df, "test")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("\n── Sanity checks ────────────────────────────────────────────────────")

# 1. Length matches split counts in windows_df
for ds, split_name in [(ds_train,"train"),(ds_val,"val"),(ds_test,"test")]:
    expected = (windows_df["split"] == split_name).sum()
    assert len(ds) == expected, \
        f"Length mismatch for {split_name}: {len(ds)} vs {expected}"
print("  ✓ Dataset lengths match windows_df split counts")

# 2. __getitem__ returns correct shape and dtype
r_sample = ds_train[0]
assert r_sample.shape  == (15,),          f"Bad shape: {r_sample.shape}"
assert r_sample.dtype  == torch.float32,  f"Bad dtype: {r_sample.dtype}"
print(f"  ✓ __getitem__ returns shape {r_sample.shape}  dtype {r_sample.dtype}")

# 3. Residual equals emp − par
row0      = ds_train._df.iloc[0]
emp_check = row0[[f"hist_emp_{k:02d}" for k in range(15)]].values.astype(np.float32)
par_check = row0[[f"hist_par_{k:02d}" for k in range(15)]].values.astype(np.float32)
expected_r = emp_check - par_check
assert np.allclose(r_sample.numpy(), expected_r, atol=1e-6), \
    "Residual mismatch: __getitem__ != emp - par"
print("  ✓ Residual = hist_emp − hist_par (verified on first item)")

# 4. No NaNs
for ds, name in [(ds_train,"train"),(ds_val,"val"),(ds_test,"test")]:
    n_nan = np.isnan(ds.all_residuals()).sum()
    assert n_nan == 0, f"NaNs found in {name}: {n_nan}"
print("  ✓ No NaNs in any split")

# 5. Train and val residuals are disjoint by cycle
train_cycles = set(zip(ds_train._df["cycle"], ds_train._df["hemisphere"]))
val_cycles   = set(zip(ds_val._df["cycle"],   ds_val._df["hemisphere"]))
test_cycles  = set(zip(ds_test._df["cycle"],  ds_test._df["hemisphere"]))
assert len(train_cycles & val_cycles)  == 0, "Train/val cycle leak!"
assert len(train_cycles & test_cycles) == 0, "Train/test cycle leak!"
assert len(val_cycles   & test_cycles) == 0, "Val/test cycle leak!"
print("  ✓ No cycle-level leakage between splits")

# ── Visualisation: residual distribution across splits ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: histogram of all residual values per split
for ds, name, color in [
        (ds_train, "Train", "tab:blue"),
        (ds_val,   "Val",   "tab:orange"),
        (ds_test,  "Test",  "tab:green")]:
    axes[0].hist(ds.all_residuals().flatten(), bins=60,
                 color=color, alpha=0.55, density=True,
                 label=f"{name} (N={len(ds)})")
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_xlabel("Residual density value")
axes[0].set_ylabel("Probability density")
axes[0].set_title("Distribution of residual values by split\n"
                   "Splits should overlap — same underlying distribution")
axes[0].legend()

# Right: mean residual profile per split
for ds, name, color, ls in [
        (ds_train, "Train", "tab:blue",   "-"),
        (ds_val,   "Val",   "tab:orange", "--"),
        (ds_test,  "Test",  "tab:green",  ":")]:
    mean_r = ds.all_residuals().mean(axis=0)
    axes[1].plot(LAT_CENTERS, mean_r, color=color, linewidth=2,
                 linestyle=ls, label=f"{name}")
    axes[1].fill_between(
        LAT_CENTERS,
        mean_r - ds.all_residuals().std(axis=0),
        mean_r + ds.all_residuals().std(axis=0),
        color=color, alpha=0.12
    )
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Latitude (°)")
axes[1].set_ylabel("Mean residual density")
axes[1].set_title("Mean residual profile by split  (± 1 std shaded)\n"
                   "Profiles should look similar — stratification check")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n✓ Task 33 complete")
print(f"  ds_train : {len(ds_train)} windows")
print(f"  ds_val   : {len(ds_val)} windows")
print(f"  ds_test  : {len(ds_test)} windows")
print(f"  Each __getitem__ returns: torch.float32 tensor of shape (15,)")
print(f"  Noise injection happens in the Lightning training step — not here.")


---
## Task 34 — Test and visualize the dataset

A dataset class is one of the easier objects to silently misimplement
because most bugs do not raise — they just return wrongly-shaped or
wrongly-typed tensors that surface as cryptic errors three layers deeper
in the training loop. Spend a few cells here on visual and numerical
sanity checks before moving on.

**Tasks:**
- Instantiate three datasets: `train_dataset`, `val_dataset`, `test_dataset`,
  using the `split` column. Print `len(...)` for each. The counts should
  match the values printed by the setup cell.
- Pull one sample from `train_dataset` (e.g. `train_dataset[0]`) and verify:
  - Its type is `torch.Tensor`.
  - Its shape is `(15,)`.
  - Its dtype is `torch.float32`.
  - It contains no `NaN` or `Inf` values.
- Pick four random rows from the training set and overplot them as bar
  charts on a 2 × 2 grid using `BIN_CENTERS` for the x-axis. They should
  look like residuals: roughly zero-mean across bins (because empirical
  and parametric densities both integrate to ~1 on the same grid), small
  at the edge bins (0–3° and 42–45°, near-empty in the data), and
  structured in the middle bins where the Spörer zone lives.
- Compute and print the bin-wise mean and standard deviation across the
  full training set. The bin-wise mean should be small (a few percent of
  density at most); the bin-wise standard deviation should be larger,
  reflecting the cycle-to-cycle structure the diffusion model will need
  to learn.


In [ ]:
# Put your code here for Task 34.
# Task 34: Test and visualize the dataset
# Depends on: ds_train, ds_val, ds_test (from Task 33)
#             LAT_CENTERS, BIN_WIDTH, N_BINS (from setup)

import torch
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: lengths ───────────────────────────────────────────────────────
print("── Dataset lengths ──────────────────────────────────────────────────")
for ds, name in [(ds_train, "train"), (ds_val, "val"), (ds_test, "test")]:
    expected = (windows_df["split"] == name).sum()
    match    = "✓" if len(ds) == expected else "✗"
    print(f"  {match} {name:<6}: {len(ds):>4} windows  "
          f"(windows_df has {expected})")

# ── Step 2: single-sample checks ─────────────────────────────────────────
print("\n── Single-sample checks (train_dataset[0]) ──────────────────────────")
sample = ds_train[0]

checks = [
    ("type is torch.Tensor",   isinstance(sample, torch.Tensor)),
    ("shape is (15,)",         sample.shape == (15,)),
    ("dtype is torch.float32", sample.dtype == torch.float32),
    ("no NaN values",          not torch.isnan(sample).any().item()),
    ("no Inf values",          not torch.isinf(sample).any().item()),
]
for desc, passed in checks:
    print(f"  {'✓' if passed else '✗'} {desc}")

print(f"\n  sample values : {sample.numpy().round(4)}")
print(f"  sample min    : {sample.min().item():.5f}")
print(f"  sample max    : {sample.max().item():.5f}")
print(f"  sample sum×BW : {(sample.numpy() * BIN_WIDTH).sum():.5f}  "
      f"(residual should integrate to ≈ 0)")

# ── Step 3: visual inspection — four random samples ───────────────────────
rng_34  = np.random.default_rng(7)
n_train = len(ds_train)
idx4    = rng_34.integers(0, n_train, size=4)

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
axes_flat = axes.flatten()

for ax, idx in zip(axes_flat, idx4):
    r    = ds_train[int(idx)].numpy()
    meta = ds_train.metadata(int(idx))

    colors = ["tab:green" if v >= 0 else "tab:red" for v in r]
    ax.bar(LAT_CENTERS, r, width=BIN_WIDTH * 0.78,
           color=colors, alpha=0.78, edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.8)

    # Mark the universal mean latitude for context
    mu_u = meta["mu_universal"]
    ax.axvline(mu_u, color="navy", linewidth=1.2, linestyle="--",
               alpha=0.7, label=f"μ_universal = {mu_u:.1f}°")

    ax.set_title(
        f"Cycle {int(meta['cycle'])} {meta['hemisphere']}  |  "
        f"τ = {meta['tau_center']:.2f} yr\n"
        f"amplitude = {meta['amplitude']:.0f} MSH  |  "
        f"n_obs = {int(meta['n_obs'])}  |  "
        f"window {meta['window_start'].date()}",
        fontsize=8
    )
    ax.set_xlabel("Latitude (°)", fontsize=8)
    ax.set_ylabel("Residual density", fontsize=8)
    ax.set_xlim(0, 45)
    ax.legend(fontsize=7)

    # Annotate sum × BW (should be near 0)
    integral = float((r * BIN_WIDTH).sum())
    ax.text(0.97, 0.04, f"∫r·dμ = {integral:.4f}",
            transform=ax.transAxes, ha="right", fontsize=7,
            color="gray")

fig.suptitle(
    "Four random training residuals\n"
    "Green = emp > par (more sunspots than model predicted)  |  "
    "Red = emp < par  |  Dashed = μ_universal",
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.show()

# ── Step 4: bin-wise statistics across the full training set ──────────────
R_train = ds_train.all_residuals()          # shape (N_train, 15)
R_val   = ds_val.all_residuals()
R_test  = ds_test.all_residuals()

mean_train = R_train.mean(axis=0)           # shape (15,)
std_train  = R_train.std(axis=0)
se_train   = std_train / np.sqrt(len(ds_train))

print("\n── Bin-wise statistics (training set) ───────────────────────────────")
print(f"  {'Bin':>4}  {'Center':>7}  {'Mean':>10}  {'Std':>10}  "
      f"{'Mean/Std':>10}  {'|Mean|>2SE':>10}")
print("  " + "-" * 62)
for k in range(N_BINS):
    ratio    = mean_train[k] / std_train[k] if std_train[k] > 0 else 0
    sig_flag = "YES" if abs(mean_train[k]) > 2 * se_train[k] else "   "
    print(f"  {k:>4}  {LAT_CENTERS[k]:>7.1f}°  "
          f"{mean_train[k]:>10.5f}  {std_train[k]:>10.5f}  "
          f"{ratio:>10.3f}  {sig_flag:>10}")

print(f"\n  Overall:")
print(f"    max |mean|       : {np.abs(mean_train).max():.5f}")
print(f"    mean std         : {std_train.mean():.5f}")
print(f"    std range        : [{std_train.min():.5f}, {std_train.max():.5f}]")
print(f"    max |mean| / mean std: "
      f"{np.abs(mean_train).max() / std_train.mean():.3f}  "
      f"(< 0.5 = residuals are mostly zero-mean)")

# ── Step 5: compare statistics across splits ──────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 4))

# --- Mean residual profile ---
ax = axes2[0]
for R, name, color, ls in [
        (R_train, "Train", "tab:blue",   "-"),
        (R_val,   "Val",   "tab:orange", "--"),
        (R_test,  "Test",  "tab:green",  ":")]:
    m = R.mean(axis=0)
    s = R.std(axis=0) / np.sqrt(len(R))
    ax.plot(LAT_CENTERS, m, color=color, linewidth=2,
            linestyle=ls, label=name)
    ax.fill_between(LAT_CENTERS, m - 2*s, m + 2*s,
                    color=color, alpha=0.12)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Latitude (°)"); ax.set_ylabel("Mean residual density")
ax.set_title("Mean residual profile ± 2SE\n"
             "(should be near zero — parametric model is unbiased)")
ax.legend(fontsize=8)

# --- Std profile ---
ax2 = axes2[1]
for R, name, color, ls in [
        (R_train, "Train", "tab:blue",   "-"),
        (R_val,   "Val",   "tab:orange", "--"),
        (R_test,  "Test",  "tab:green",  ":")]:
    ax2.plot(LAT_CENTERS, R.std(axis=0), color=color, linewidth=2,
             linestyle=ls, label=name)
ax2.set_xlabel("Latitude (°)"); ax2.set_ylabel("Std of residual density")
ax2.set_title("Bin-wise std across splits\n"
              "(higher std = more cycle-to-cycle variability here)")
ax2.legend(fontsize=8)

# --- Distribution of all residual values ---
ax3 = axes2[2]
for R, name, color in [
        (R_train, "Train", "tab:blue"),
        (R_val,   "Val",   "tab:orange"),
        (R_test,  "Test",  "tab:green")]:
    ax3.hist(R.flatten(), bins=50, color=color, alpha=0.5,
             density=True, label=f"{name} (N={len(R)})")
ax3.axvline(0, color="black", linewidth=1)
ax3.set_xlabel("Residual density value")
ax3.set_ylabel("Probability density")
ax3.set_title("Distribution of all residual values\n"
              "(splits should overlap — same underlying distribution)")
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Step 6: correlation structure of training residuals ────────────────────
cov_train  = np.cov(R_train.T)     # shape (15, 15)
corr_train = np.corrcoef(R_train.T)

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 5))

from matplotlib.colors import TwoSlopeNorm
norm_corr = TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

im0 = axes3[0].imshow(corr_train, cmap="RdBu_r", norm=norm_corr, aspect="auto")
axes3[0].set_title("Correlation matrix of training residuals\n"
                    "Off-diagonal structure = bins co-vary across windows")
plt.colorbar(im0, ax=axes3[0], fraction=0.046, pad=0.04)

ticks = np.arange(0, N_BINS, 3)
tick_labels = [f"{LAT_CENTERS[i]:.0f}°" for i in ticks]
for ax in axes3:
    ax.set_xticks(ticks); ax.set_xticklabels(tick_labels, fontsize=7)
    ax.set_yticks(ticks); ax.set_yticklabels(tick_labels, fontsize=7)

# Plot the leading eigenvector (dominant mode of variation)
eigvals, eigvecs = np.linalg.eigh(cov_train)
order            = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

axes3[1].bar(LAT_CENTERS, eigvecs[:, 0], width=BIN_WIDTH * 0.75,
             color=["tab:green" if v >= 0 else "tab:red"
                    for v in eigvecs[:, 0]], alpha=0.8)
axes3[1].axhline(0, color="black", linewidth=0.8)
axes3[1].set_xlabel("Latitude (°)")
axes3[1].set_ylabel("Eigenvector component")
axes3[1].set_title(
    f"Leading eigenvector of training residual covariance\n"
    f"Explains {eigvals[0]/eigvals.sum()*100:.1f}% of total variance — "
    f"dominant mode of cycle-to-cycle variation"
)

plt.tight_layout()
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────
print("\n── Summary ──────────────────────────────────────────────────────────")
print(f"  train: {len(ds_train)} windows  "
      f"| mean abs residual: {np.abs(R_train).mean():.5f}")
print(f"  val  : {len(ds_val)} windows  "
      f"| mean abs residual: {np.abs(R_val).mean():.5f}")
print(f"  test : {len(ds_test)} windows  "
      f"| mean abs residual: {np.abs(R_test).mean():.5f}")
print(f"\n  Leading eigenvalue explains "
      f"{eigvals[0]/eigvals.sum()*100:.1f}% of training variance")
print(f"  → diffusion model needs to learn this dominant mode "
      f"plus the remaining {eigvals[1:].sum()/eigvals.sum()*100:.1f}%")
print(f"\n✓ Task 34 complete — dataset verified, ready for DataLoaders (Task 35)")


---
## Task 35 — Build the DataLoaders

A `Dataset` is an indexed collection; a `DataLoader` is the object that
samples batches from it during training. The DataLoader's job is to handle
batching, shuffling, parallel data loading, and the iteration protocol the
Lightning trainer expects.

There is one design choice worth flagging that is not diffusion-specific
but is easy to get wrong: `shuffle=True` for the *training* loader and
`shuffle=False` for validation and test. The reason for shuffling at
training time is that stochastic gradient descent assumes batches are
roughly i.i.d. samples from the data distribution; if batches are returned
in a fixed order (especially if the data is sorted by cycle, as our
parquet roughly is), the gradient is biased per epoch and the model can
fit cycle-by-cycle artifacts. For validation and test, the order does not
affect the loss value; we leave shuffling off so that successive
validation runs produce identical batch orderings, which keeps val/test
metrics reproducible across epochs.

**Tasks:**
- Build three DataLoaders using `torch.utils.data.DataLoader`:
  - `train_loader`: `batch_size=64`, `shuffle=True`, `num_workers=0`
    (Colab is happiest with 0 here).
  - `val_loader`: `batch_size=64`, `shuffle=False`, `num_workers=0`.
  - `test_loader`: `batch_size=64`, `shuffle=False`, `num_workers=0`.
- Iterate one batch from each loader (`next(iter(loader))`) and verify:
  - The batch is a tensor of shape `(64, 15)` (or smaller for the last
    batch if the dataset size is not divisible by 64).
  - The dtype is `torch.float32`.
- Print the number of batches per epoch for each loader (`len(loader)`).

**Note for later:** with 15-dim data and batch size 64, a full pass through
the training set is fast — well under a second on Colab's GPU. We will
lean into this in Task 41 by training for many epochs.


In [ ]:
# Put your code here for Task 35.
# Task 35: Build the DataLoaders
# Depends on: ds_train, ds_val, ds_test (from Task 33)

from torch.utils.data import DataLoader

# ── Build the three loaders ───────────────────────────────────────────────
BATCH_SIZE = 64

train_loader = DataLoader(
    ds_train,
    batch_size  = BATCH_SIZE,
    shuffle     = True,       # essential for SGD — randomizes batch order
    num_workers = 0,          # 0 = main process only (Colab-safe)
    pin_memory  = DEVICE == "cuda",  # speeds up CPU→GPU transfer if on GPU
    drop_last   = False,      # keep the final partial batch
)

val_loader = DataLoader(
    ds_val,
    batch_size  = BATCH_SIZE,
    shuffle     = False,      # reproducible ordering for metrics
    num_workers = 0,
    pin_memory  = DEVICE == "cuda",
    drop_last   = False,
)

test_loader = DataLoader(
    ds_test,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = 0,
    pin_memory  = DEVICE == "cuda",
    drop_last   = False,
)

# ── Batch counts ──────────────────────────────────────────────────────────
print("── DataLoader summary ───────────────────────────────────────────────")
for loader, name in [
        (train_loader, "train"),
        (val_loader,   "val"),
        (test_loader,  "test")]:
    n_windows = len(loader.dataset)
    n_batches = len(loader)
    last_bs   = n_windows % BATCH_SIZE or BATCH_SIZE
    print(f"  {name:<6}: {n_windows:>4} windows  →  {n_batches:>3} batches/epoch  "
          f"(last batch size: {last_bs})")

# ── Step-through one batch from each loader ────────────────────────────────
print("\n── Batch verification ───────────────────────────────────────────────")
for loader, name in [
        (train_loader, "train"),
        (val_loader,   "val"),
        (test_loader,  "test")]:

    batch = next(iter(loader))

    # Expected shape: (min(BATCH_SIZE, len(dataset)), 15)
    expected_bs = min(BATCH_SIZE, len(loader.dataset))
    shape_ok    = (batch.shape[0] <= BATCH_SIZE and batch.shape[1] == 15)
    dtype_ok    = (batch.dtype == torch.float32)
    nan_ok      = not torch.isnan(batch).any().item()
    inf_ok      = not torch.isinf(batch).any().item()

    status = "✓" if all([shape_ok, dtype_ok, nan_ok, inf_ok]) else "✗"
    print(f"  {status} {name:<6}: shape={tuple(batch.shape)}  "
          f"dtype={batch.dtype}  "
          f"NaN={not nan_ok}  Inf={not inf_ok}")
    print(f"         min={batch.min().item():.5f}  "
          f"max={batch.max().item():.5f}  "
          f"mean={batch.mean().item():.5f}")

# ── Verify shuffle is working: two consecutive train batches differ ────────
print("\n── Shuffle verification (train loader) ──────────────────────────────")
it        = iter(train_loader)
batch_a   = next(it)
batch_b   = next(it)
identical = torch.equal(batch_a, batch_b)
print(f"  Consecutive batches identical: {identical}  "
      f"(should be False — shuffle is working if False)")

# ── Verify val loader is deterministic across two full passes ─────────────
print("\n── Determinism verification (val loader) ────────────────────────────")
first_pass  = torch.cat([b for b in val_loader], dim=0)
second_pass = torch.cat([b for b in val_loader], dim=0)
det_ok      = torch.equal(first_pass, second_pass)
print(f"  Two full val passes identical: {det_ok}  "
      f"(should be True — shuffle=False guarantees this)")

# ── Simulate what the training step receives ──────────────────────────────
print("\n── Training step simulation ─────────────────────────────────────────")
print("  The Lightning training_step receives one batch r of shape (B, 15).")
print("  Inside the step it will:")
print("    1. sample t  ~ Uniform{0, …, T}  — one per row  → shape (B,)")
print("    2. draw   ε  ~ N(0, I)           — one per row  → shape (B, 15)")
print("    3. compute r_t = α[t]·r + σ[t]·ε               → shape (B, 15)")
print("    4. predict ε̂  = network(r_t, t)                → shape (B, 15)")
print("    5. loss = MSE(ε̂, ε)")
print()

# Show what a simulated training step would look like with a real batch
r_batch = next(iter(train_loader))           # shape (B, 15)
B       = r_batch.shape[0]

# Sample random timesteps
rng_35  = torch.Generator(); rng_35.manual_seed(0)
t_batch = torch.randint(0, T + 1, (B,), generator=rng_35)  # shape (B,)

# Draw noise
eps_batch = torch.randn(B, 15)                              # shape (B, 15)

# Corrupt: need α[t] and σ[t] per row — shape (B, 1) for broadcasting
alpha_t = alpha_pt[t_batch].unsqueeze(1)                   # shape (B, 1)
sigma_t = sigma_pt[t_batch].unsqueeze(1)                   # shape (B, 1)
r_t     = alpha_t * r_batch + sigma_t * eps_batch          # shape (B, 15)

print(f"  Simulated training batch:")
print(f"    r      : {tuple(r_batch.shape)}  "
      f"range [{r_batch.min():.4f}, {r_batch.max():.4f}]")
print(f"    t      : {tuple(t_batch.shape)}  "
      f"range [{t_batch.min().item()}, {t_batch.max().item()}]")
print(f"    ε      : {tuple(eps_batch.shape)}  "
      f"mean {eps_batch.mean():.4f}  std {eps_batch.std():.4f}")
print(f"    r_t    : {tuple(r_t.shape)}  "
      f"range [{r_t.min():.4f}, {r_t.max():.4f}]")
print(f"    target : predict ε from (r_t, t) → MSE loss")

# ── Visualisation: one training batch ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Left: distribution of t values in one batch
axes[0].hist(t_batch.numpy(), bins=20, color="steelblue", alpha=0.75,
             edgecolor="white")
axes[0].set_xlabel("Sampled timestep t")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Timestep distribution in one batch (B={B})\n"
                   "Uniform over [0, T] — every noise level seen equally")

# Middle: α[t] and σ[t] values for this batch
axes[1].scatter(t_batch.numpy(), alpha_pt[t_batch].numpy(),
                s=12, alpha=0.6, color="tab:green", label="α_t")
axes[1].scatter(t_batch.numpy(), sigma_pt[t_batch].numpy(),
                s=12, alpha=0.6, color="tab:red",   label="σ_t")
axes[1].set_xlabel("Timestep t")
axes[1].set_ylabel("Schedule coefficient")
axes[1].set_title("α_t and σ_t for each batch element\n"
                   "Each row gets its own corruption level")
axes[1].legend(fontsize=8)

# Right: one clean vs one corrupted residual from the batch
k_show = 0
axes[2].bar(LAT_CENTERS - 0.6, r_batch[k_show].numpy(),
            width=BIN_WIDTH * 0.42, color="steelblue", alpha=0.75,
            label=f"r₀  (clean)")
axes[2].bar(LAT_CENTERS + 0.6, r_t[k_show].detach().numpy(),
            width=BIN_WIDTH * 0.42, color="tab:orange", alpha=0.75,
            label=f"r_t  (t={t_batch[k_show].item()}, "
                  f"SNR={snr[t_batch[k_show].item()]:.2f})")
axes[2].axhline(0, color="black", linewidth=0.7)
axes[2].set_xlabel("Latitude (°)")
axes[2].set_ylabel("Residual density")
axes[2].set_title("First batch element: clean vs corrupted\n"
                   "(what the network receives vs what it started from)")
axes[2].legend(fontsize=7.5)
axes[2].set_xlim(0, 45)

plt.tight_layout()
plt.show()

print(f"\n✓ Task 35 complete")
print(f"  train_loader: {len(train_loader)} batches/epoch  "
      f"(shuffle=True)")
print(f"  val_loader  : {len(val_loader)} batches/epoch  "
      f"(shuffle=False, deterministic)")
print(f"  test_loader : {len(test_loader)} batches/epoch  "
      f"(shuffle=False, deterministic)")
print(f"\n  Ready for Task 36 — sinusoidal timestep embedding")


---
## Task 36 — Implement the sinusoidal timestep embedding

We arrive at the first diffusion-specific piece of architecture. The model
trained in this notebook is a single network that has to handle every
noise level from t = 0 (clean data) to t = T-1 (essentially pure noise) —
200 different denoising tasks, all sharing weights. For one network to
behave differently at different noise levels, it needs to *know* which t
it is currently denoising. The mechanism that gives it that information
is called a **timestep embedding**: a learned representation of the integer
t that is fed into the network alongside r_t.

The standard recipe — borrowed wholesale from how Transformers encode
positions — is the **sinusoidal embedding**. Given an integer t and an
embedding dimension d, produce a d-dimensional vector whose components are
sines and cosines of t at logarithmically spaced frequencies. Different t
values produce vectors that point in different directions in this
d-dimensional space; the geometry is smooth (nearby t produce nearby
embeddings) and the frequencies span a wide range so that both fast and
slow variation in t can be represented.

The function below produces the raw sinusoidal embedding. Task 37 will
wrap it in a small learnable MLP. The embedding itself is fixed (no
learnable parameters); only the wrapper learns.

**Tasks:**
- Implement `sinusoidal_embedding(t, dim)` with the following contract:
  - `t` is a `torch.Tensor` of shape `(B,)` containing integer timesteps.
  - `dim` is the desired embedding dimension (an even integer; we will use
    64 in Task 37).
  - The function returns a `torch.Tensor` of shape `(B, dim)`.
- Use `einops.rearrange` for the broadcast that combines `t` and the
  frequency vector into a `(B, dim/2)` argument tensor. Do not use
  `unsqueeze`, `reshape`, or `view`.
- The frequency formula is `freqs[i] = exp(-log(10000) * i / (dim/2))` for
  `i` in `0, 1, …, dim/2 - 1`. The argument tensor is then
  `args[b, i] = t[b] * freqs[i]`. The embedding stacks `sin(args)` and
  `cos(args)` along the feature dimension.

**Visualization tasks:**
- Pick `dim=64` and compute the embedding for `t = [0, T//4, T//2, 3*T//4, T-1]`.
  Plot the five 64-dimensional vectors as bar charts on a single figure
  (one row per t value). The patterns should differ visibly — that visible
  difference is what allows a downstream MLP to behave differently at
  different t.
- For three chosen embedding dimensions (e.g. dim 0, 16, 32), plot the
  embedding value as a function of t for `t` ranging over `[0, T)`. Each
  plot should look like a sinusoid; the wavelength should grow with the
  dimension index, because higher dimension indices correspond to lower
  frequencies in this formula.


In [ ]:
# Put your code here for Task 36.
# Task 36: Sinusoidal timestep embedding
# Depends on: T, schedule (from setup), einops

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange, repeat

def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Sinusoidal timestep embedding (fixed, no learnable parameters).

    Parameters
    ----------
    t   : torch.Tensor, shape (B,)  — integer timesteps
    dim : int  — embedding dimension, must be even

    Returns
    -------
    emb : torch.Tensor, shape (B, dim)
          First dim/2 components are sin, last dim/2 are cos.
    """
    assert dim % 2 == 0, f"dim must be even, got {dim}"
    assert t.ndim == 1,  f"t must be shape (B,), got {t.shape}"

    half = dim // 2

    # Frequency vector: shape (half,)
    # freqs[i] = exp(-log(10000) * i / half)
    # i=0 → freq=1 (fastest),  i=half-1 → freq=10000^{-1} (slowest)
    i     = torch.arange(half, dtype=torch.float32, device=t.device)
    freqs = torch.exp(-np.log(10000.0) * i / half)   # shape (half,)

    # Outer product t × freqs → args of shape (B, half)
    # Use einops: t is (B,), freqs is (half,)
    # rearrange t to (B, 1) and freqs to (1, half), then multiply
    t_col    = rearrange(t.float(), "b -> b 1")       # (B, 1)
    freq_row = rearrange(freqs,     "h -> 1 h")       # (1, half)
    args     = t_col * freq_row                        # (B, half)

    # Stack sin and cos along the feature dimension
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  # (B, dim)
    return emb


# ── Quick shape + dtype check ─────────────────────────────────────────────
DIM_EMB  = 64
t_test   = torch.tensor([0, T//4, T//2, 3*T//4, T-1])
emb_test = sinusoidal_embedding(t_test, DIM_EMB)

print("── Shape and dtype checks ───────────────────────────────────────────")
print(f"  Input  t : {tuple(t_test.shape)}  values: {t_test.tolist()}")
print(f"  Output   : {tuple(emb_test.shape)}  dtype: {emb_test.dtype}")
assert emb_test.shape == (5, DIM_EMB), f"Bad shape: {emb_test.shape}"
assert emb_test.dtype == torch.float32
print("  ✓ shape (5, 64)  ✓ dtype float32")

# Check that different t values produce different embeddings
for i in range(len(t_test)):
    for j in range(i+1, len(t_test)):
        assert not torch.equal(emb_test[i], emb_test[j]), \
            f"t={t_test[i]} and t={t_test[j]} produced identical embeddings!"
print("  ✓ all five t values produce distinct embeddings")

# Check range: sin/cos are bounded in [-1, 1]
print(f"  Embedding range: [{emb_test.min().item():.4f}, "
      f"{emb_test.max().item():.4f}]  (should be within [-1, 1])")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: Five embeddings as bar charts (one row per t)
# ══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(5, 1, figsize=(16, 10), sharex=True)

t_labels  = [f"t = {v.item()}  ({v.item()/T*100:.0f}% through schedule)"
             for v in t_test]
dim_range = np.arange(DIM_EMB)

for ax, emb_row, label, t_val in zip(
        axes1, emb_test.numpy(), t_labels, t_test.tolist()):
    ax.bar(dim_range[:DIM_EMB//2], emb_row[:DIM_EMB//2],
           color="tab:blue", alpha=0.7, width=0.8, label="sin components")
    ax.bar(dim_range[DIM_EMB//2:], emb_row[DIM_EMB//2:],
           color="tab:orange", alpha=0.7, width=0.8, label="cos components")
    ax.axhline(0,  color="black", linewidth=0.6)
    ax.axvline(DIM_EMB//2 - 0.5, color="gray",
               linewidth=1, linestyle="--", alpha=0.5)
    ax.set_ylim(-1.1, 1.1)
    ax.set_ylabel("Value", fontsize=7)
    ax.set_title(
        f"{label}  |  SNR = {snr[t_val]:.3f}",
        fontsize=8
    )
    if ax == axes1[0]:
        ax.legend(fontsize=7, loc="upper right")

axes1[-1].set_xlabel("Embedding dimension index")
axes1[-1].set_xticks(np.arange(0, DIM_EMB, 8))

fig1.suptitle(
    f"Sinusoidal timestep embeddings (dim={DIM_EMB})\n"
    "Blue: sin components (dims 0–31)   Orange: cos components (dims 32–63)\n"
    "Each row corresponds to a different t — the patterns must differ "
    "visibly for the network to distinguish noise levels",
    fontsize=9, y=1.01
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: Embedding value as a function of t for selected dimensions
# ══════════════════════════════════════════════════════════════════════════
t_all   = torch.arange(0, T, dtype=torch.long)    # shape (T,)
emb_all = sinusoidal_embedding(t_all, DIM_EMB)    # shape (T, 64)

# Pick three sin dimensions and three cos dimensions
sin_dims = [0, 8, 16]   # fast, medium, slow
cos_dims = [32, 40, 48]
chosen_dims = sin_dims + cos_dims
half        = DIM_EMB // 2

fig2, axes2 = plt.subplots(2, 3, figsize=(15, 6), sharey=True)
axes2_flat  = axes2.flatten()

for ax, d in zip(axes2_flat, chosen_dims):
    component = "sin" if d < half else "cos"
    freq_idx  = d if d < half else d - half
    freq_val  = np.exp(-np.log(10000.0) * freq_idx / half)
    wavelength_in_t = 2 * np.pi / freq_val

    ax.plot(t_all.numpy(), emb_all[:, d].numpy(),
            color="tab:blue" if d < half else "tab:orange",
            linewidth=1.5, alpha=0.9)
    ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
    ax.set_xlabel("Timestep t", fontsize=8)
    ax.set_ylabel("Embedding value", fontsize=8)
    ax.set_title(
        f"Dim {d}  ({component}[{freq_idx}])\n"
        f"freq = {freq_val:.5f}   wavelength ≈ {wavelength_in_t:.1f} steps",
        fontsize=8
    )
    ax.set_xlim(0, T)

fig2.suptitle(
    "Embedding value vs timestep t for selected dimensions\n"
    "Low dim index → high frequency (fast oscillation)   "
    "High dim index → low frequency (slow drift)\n"
    "Wavelength grows with dim index — this spans all timescales "
    "from fine-grained (dim 0) to coarse (dim 48)",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: Embedding matrix heatmap + pairwise distance
# ══════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(1, 2, figsize=(15, 5))

# Left: heatmap of the full (T, dim) embedding matrix
im = axes3[0].imshow(
    emb_all.numpy().T,     # shape (dim, T) so t is on x-axis
    aspect="auto", cmap="RdBu_r",
    vmin=-1, vmax=1,
    extent=[0, T, DIM_EMB, 0]
)
axes3[0].axhline(half, color="white", linewidth=1.5, linestyle="--",
                 alpha=0.7)
axes3[0].set_xlabel("Timestep t")
axes3[0].set_ylabel("Embedding dimension")
axes3[0].set_title(
    f"Full embedding matrix  (shape T × dim = {T} × {DIM_EMB})\n"
    "White dashed line separates sin (top) from cos (bottom) components"
)
plt.colorbar(im, ax=axes3[0], fraction=0.046, pad=0.04)

# Right: pairwise L2 distance between selected t embeddings
t_sample    = torch.arange(0, T, T//20)   # 20 evenly spaced timesteps
emb_sample  = sinusoidal_embedding(t_sample, DIM_EMB)
dists       = torch.cdist(emb_sample, emb_sample).numpy()

im2 = axes3[1].imshow(dists, aspect="auto", cmap="viridis")
axes3[1].set_title(
    "Pairwise L2 distance between embeddings\n"
    "Diagonal = 0,  off-diagonal should grow with |t_i − t_j|\n"
    "(smooth geometry = nearby t → nearby embeddings)"
)
n_sample = len(t_sample)
tick_step = max(1, n_sample // 5)
axes3[1].set_xticks(np.arange(0, n_sample, tick_step))
axes3[1].set_xticklabels(t_sample[::tick_step].numpy(), fontsize=7)
axes3[1].set_yticks(np.arange(0, n_sample, tick_step))
axes3[1].set_yticklabels(t_sample[::tick_step].numpy(), fontsize=7)
axes3[1].set_xlabel("Timestep t")
axes3[1].set_ylabel("Timestep t")
plt.colorbar(im2, ax=axes3[1], fraction=0.046, pad=0.04, label="L2 distance")

plt.tight_layout()
plt.show()

# ── Numeric summary ────────────────────────────────────────────────────────
print("\n── Numeric summary ──────────────────────────────────────────────────")
print(f"  dim = {DIM_EMB}  →  {half} sin dims + {half} cos dims")
print(f"  Frequency range: [{np.exp(-np.log(10000)*0/half):.4f}, "
      f"{np.exp(-np.log(10000)*(half-1)/half):.6f}]")
print(f"  Wavelength range: [{2*np.pi/np.exp(-np.log(10000)*0/half):.1f}, "
      f"{2*np.pi/np.exp(-np.log(10000)*(half-1)/half):.0f}] timesteps")
print(f"  T = {T} timesteps — longest wavelength >> T means the "
      f"slowest dimensions don't even complete one cycle")

# Mean pairwise distance between adjacent vs distant timesteps
adj_dists  = [float(torch.dist(sinusoidal_embedding(torch.tensor([t_]),   DIM_EMB),
                                sinusoidal_embedding(torch.tensor([t_+1]), DIM_EMB)))
              for t_ in range(0, T-1, 10)]
far_dists  = [float(torch.dist(sinusoidal_embedding(torch.tensor([t_]),       DIM_EMB),
                                sinusoidal_embedding(torch.tensor([t_+T//4]), DIM_EMB)))
              for t_ in range(0, 3*T//4, 10)]
print(f"\n  Mean L2 dist between adjacent t    : {np.mean(adj_dists):.4f}")
print(f"  Mean L2 dist between t and t+T//4 : {np.mean(far_dists):.4f}")
print(f"  Ratio (far/adj)                    : "
      f"{np.mean(far_dists)/np.mean(adj_dists):.2f}×  "
      f"(larger = more discriminable)")

print(f"\n✓ Task 36 complete — sinusoidal_embedding(t, dim) ready")
print(f"  Signature : sinusoidal_embedding(t: Tensor[B], dim: int) → Tensor[B, dim]")
print(f"  No learnable parameters — fixed geometric encoding of t")
print(f"  Task 37 will wrap this in a small MLP to make it learnable")

---
## Task 37 — Wrap the sinusoidal embedding in a learnable module

The raw sinusoidal embedding is fixed: it has no learnable parameters and
its representation of t is determined entirely by the formula. To let the
main network shape the t-representation it actually wants, we wrap the
sinusoidal embedding in a small MLP that *learns* to project the fixed
sinusoidal representation into a useful form. This wrapper is the
`TimestepEmbedding` module.

The wrapping pattern (fixed positional encoding → learnable projection)
is standard across Transformer-style architectures. It separates the
"how do I represent integer position as a vector" question (solved
analytically by the sinusoidal formula) from the "what t-information does
my downstream network find useful" question (solved by gradient descent
through the MLP).

**Tasks:**
- Implement `TimestepEmbedding(nn.Module)` with constructor arguments
  `embed_dim` (the dimension of the raw sinusoidal embedding, 64 by
  default) and `hidden_dim` (the dimension of the learned projection,
  128 by default).
- The constructor should build a small MLP:
  `Linear(embed_dim, hidden_dim) → SiLU → Linear(hidden_dim, hidden_dim)`.
  We use SiLU (also called Swish: x · sigmoid(x)) because it is the
  conventional activation in diffusion models and gives smoother gradients
  than ReLU at deep noise levels. ReLU also works; SiLU is just the
  standard pick.
- The `forward(self, t)` method takes `t` of shape `(B,)`, calls
  `sinusoidal_embedding(t, self.embed_dim)`, and returns the result of the
  MLP, of shape `(B, hidden_dim)`.

**Sanity check:** instantiate `TimestepEmbedding(embed_dim=64, hidden_dim=128)`,
construct a tensor `t = torch.arange(0, T, T // 8)` (shape `(8,)`), and
verify that `module(t)` returns a tensor of shape `(8, 128)` with no NaNs.


In [ ]:
# Put your code here for Task 37.
# Task 37: TimestepEmbedding module
# Depends on: sinusoidal_embedding (Task 36), torch, einops

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from einops import rearrange

class TimestepEmbedding(nn.Module):
    """
    Wraps the fixed sinusoidal embedding in a small learnable MLP.

    Architecture:
        sinusoidal_embedding(t, embed_dim)        → (B, embed_dim)   [fixed]
        Linear(embed_dim, hidden_dim) → SiLU      → (B, hidden_dim)  [learned]
        Linear(hidden_dim, hidden_dim)             → (B, hidden_dim)  [learned]

    Parameters
    ----------
    embed_dim  : int  — dimension of the raw sinusoidal embedding (default 64)
    hidden_dim : int  — output dimension of the learned projection (default 128)
    """

    def __init__(self, embed_dim: int = 64, hidden_dim: int = 128):
        super().__init__()
        assert embed_dim % 2 == 0, f"embed_dim must be even, got {embed_dim}"

        self.embed_dim  = embed_dim
        self.hidden_dim = hidden_dim

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.SiLU(),                          # Swish: x · σ(x)
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        t : torch.Tensor, shape (B,)  — integer timesteps

        Returns
        -------
        emb : torch.Tensor, shape (B, hidden_dim)
        """
        # Fixed sinusoidal encoding — no gradient flows through this
        sin_emb = sinusoidal_embedding(t, self.embed_dim)   # (B, embed_dim)

        # Learnable projection — gradient flows through this
        return self.mlp(sin_emb)                             # (B, hidden_dim)

    def extra_repr(self) -> str:
        return (f"embed_dim={self.embed_dim}, "
                f"hidden_dim={self.hidden_dim}, "
                f"params={sum(p.numel() for p in self.parameters()):,}")


# ── Instantiate and inspect ────────────────────────────────────────────────
EMBED_DIM  = 64
HIDDEN_DIM = 128

t_emb_module = TimestepEmbedding(embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM)
print("── Module summary ───────────────────────────────────────────────────")
print(t_emb_module)
print(f"\nParameter count: "
      f"{sum(p.numel() for p in t_emb_module.parameters()):,}")
for name, p in t_emb_module.named_parameters():
    print(f"  {name:<30} shape={tuple(p.shape)}  "
          f"numel={p.numel()}")

# ── Sanity checks ─────────────────────────────────────────────────────────
print("\n── Sanity checks ────────────────────────────────────────────────────")
t_sanity = torch.arange(0, T, T // 8)   # shape (8,)
print(f"  Input t : {tuple(t_sanity.shape)}  values: {t_sanity.tolist()}")

t_emb_module.eval()
with torch.no_grad():
    out_sanity = t_emb_module(t_sanity)

checks = [
    ("output shape is (8, 128)",   out_sanity.shape == (8, HIDDEN_DIM)),
    ("output dtype is float32",    out_sanity.dtype == torch.float32),
    ("no NaN in output",           not torch.isnan(out_sanity).any().item()),
    ("no Inf in output",           not torch.isinf(out_sanity).any().item()),
    ("different t → different emb",
     all(not torch.equal(out_sanity[i], out_sanity[j])
         for i in range(8) for j in range(i+1, 8))),
]
for desc, passed in checks:
    print(f"  {'✓' if passed else '✗'} {desc}")

print(f"\n  Output shape : {tuple(out_sanity.shape)}")
print(f"  Output range : [{out_sanity.min().item():.4f}, "
      f"{out_sanity.max().item():.4f}]")
print(f"  Output mean  : {out_sanity.mean().item():.4f}")
print(f"  Output std   : {out_sanity.std().item():.4f}")

# ── Verify gradient flows through the MLP but not the sinusoidal part ─────
print("\n── Gradient flow check ──────────────────────────────────────────────")
t_grad = torch.arange(0, T, T // 8)
out_grad = t_emb_module(t_grad)
loss_dummy = out_grad.sum()
loss_dummy.backward()

for name, p in t_emb_module.named_parameters():
    has_grad = p.grad is not None
    grad_norm = p.grad.norm().item() if has_grad else 0.0
    print(f"  {name:<30} grad={'✓' if has_grad else '✗'}  "
          f"norm={grad_norm:.4f}")

# ══════════════════════════════════════════════════════════════════════════
# Figure 1: raw sinusoidal vs learned projection — side by side
# ══════════════════════════════════════════════════════════════════════════
t_all = torch.arange(0, T, dtype=torch.long)

# Raw sinusoidal (shape T, embed_dim)
with torch.no_grad():
    sin_all = sinusoidal_embedding(t_all, EMBED_DIM)
    mlp_all = t_emb_module(t_all)              # shape (T, hidden_dim)

fig1, axes1 = plt.subplots(1, 2, figsize=(15, 5))

im0 = axes1[0].imshow(
    sin_all.numpy().T, aspect="auto", cmap="RdBu_r",
    vmin=-1, vmax=1, extent=[0, T, EMBED_DIM, 0]
)
axes1[0].set_title(
    f"Raw sinusoidal embedding  (fixed, no parameters)\n"
    f"Shape: T × embed_dim = {T} × {EMBED_DIM}",
    fontsize=9
)
axes1[0].set_xlabel("Timestep t")
axes1[0].set_ylabel("Embedding dimension")
plt.colorbar(im0, ax=axes1[0], fraction=0.046, pad=0.04)

vmax_mlp = mlp_all.abs().quantile(0.98).item()
im1 = axes1[1].imshow(
    mlp_all.detach().numpy().T, aspect="auto", cmap="RdBu_r",
    vmin=-vmax_mlp, vmax=vmax_mlp, extent=[0, T, HIDDEN_DIM, 0]
)
axes1[1].set_title(
    f"Learned MLP projection  (random init — will change after training)\n"
    f"Shape: T × hidden_dim = {T} × {HIDDEN_DIM}",
    fontsize=9
)
axes1[1].set_xlabel("Timestep t")
axes1[1].set_ylabel("Hidden dimension")
plt.colorbar(im1, ax=axes1[1], fraction=0.046, pad=0.04)

fig1.suptitle(
    "TimestepEmbedding: fixed sinusoidal input → learned MLP output\n"
    "After training the right panel will encode the t-information "
    "the denoising network finds most useful",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 2: pairwise distances — does learning preserve sinusoidal geometry?
# ══════════════════════════════════════════════════════════════════════════
t_sample   = torch.arange(0, T, T // 20)
with torch.no_grad():
    sin_samp = sinusoidal_embedding(t_sample, EMBED_DIM)
    mlp_samp = t_emb_module(t_sample)

dist_sin = torch.cdist(sin_samp, sin_samp).numpy()
dist_mlp = torch.cdist(mlp_samp, mlp_samp).numpy()

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
for ax, dmat, title in [
        (axes2[0], dist_sin, "Pairwise L2: raw sinusoidal\n(fixed)"),
        (axes2[1], dist_mlp, "Pairwise L2: MLP projection\n(random init)")]:
    im = ax.imshow(dmat, aspect="auto", cmap="viridis")
    ax.set_title(title, fontsize=9)
    n = len(t_sample)
    step = max(1, n // 5)
    ax.set_xticks(np.arange(0, n, step))
    ax.set_xticklabels(t_sample[::step].numpy(), fontsize=7)
    ax.set_yticks(np.arange(0, n, step))
    ax.set_yticklabels(t_sample[::step].numpy(), fontsize=7)
    ax.set_xlabel("Timestep t"); ax.set_ylabel("Timestep t")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="L2 distance")

fig2.suptitle(
    "Pairwise distance structure before training\n"
    "Left (fixed): smooth gradient from diagonal outward — good geometry.\n"
    "Right (random init): scrambled — training will reshape this to be useful.",
    fontsize=9
)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════
# Figure 3: SiLU vs ReLU activation — why SiLU?
# ══════════════════════════════════════════════════════════════════════════
x     = torch.linspace(-4, 4, 200)
silu  = torch.nn.functional.silu(x)
relu  = torch.nn.functional.relu(x)
gelu  = torch.nn.functional.gelu(x)

fig3, ax3 = plt.subplots(figsize=(8, 4))
ax3.plot(x.numpy(), silu.numpy(),  color="tab:blue",   linewidth=2,
         label="SiLU (used here)  x·σ(x)")
ax3.plot(x.numpy(), relu.numpy(),  color="tab:orange", linewidth=2,
         linestyle="--", label="ReLU  max(0, x)")
ax3.plot(x.numpy(), gelu.numpy(),  color="tab:green",  linewidth=2,
         linestyle=":", label="GELU")
ax3.axhline(0, color="black", linewidth=0.7)
ax3.axvline(0, color="black", linewidth=0.7)
ax3.set_xlabel("Input x"); ax3.set_ylabel("Activation output")
ax3.set_title(
    "SiLU vs ReLU vs GELU\n"
    "SiLU is smooth everywhere (no kink at 0) and slightly negative "
    "for x < 0\n→ smoother gradients, standard in diffusion architectures"
)
ax3.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"\n✓ Task 37 complete")
print(f"  TimestepEmbedding(embed_dim={EMBED_DIM}, hidden_dim={HIDDEN_DIM})")
print(f"  Parameters: {sum(p.numel() for p in t_emb_module.parameters()):,}")
print(f"  forward(t: Tensor[B]) → Tensor[B, {HIDDEN_DIM}]")
print(f"  Ready for Task 38 — DiffusionMLP main network")


---
## Task 38 — Build the DiffusionMLP with an ablation flag

The main network: an MLP that takes (r_t, t), concatenates the timestep
embedding with r_t at the input, pushes through a stack of fully connected
layers, and returns ε̂ — the network's prediction of the noise that was
added. The output has the same shape as r_t.

The diffusion-specific architectural concern here is *how* t enters the
network. We use **input concatenation**: the timestep embedding is
appended to r_t at the input layer, and the MLP learns to read both pieces
from the concatenated vector. Two alternatives exist that are common in
image diffusion models — FiLM (feature-wise modulation, where t scales and
shifts feature maps at every layer) and cross-attention (where t serves as
a key/value pair attended to from r_t) — but for 15-dimensional data and
a 3-layer MLP, both are overkill. Input concatenation is the right level
of complexity for the residual problem.

The constructor takes a `use_timestep_embedding=True/False` flag. When
True, the network builds a TimestepEmbedding and concatenates its output
with r_t. When False, the network has no TimestepEmbedding, the input
dimension is just `data_dim=15`, and the network is t-blind — it produces
the same output regardless of t. The False setting is the **architectural
ablation** we will use in Task 44 to measure whether the timestep
embedding actually earns its keep on this dataset.

**Tasks:**
- Implement `DiffusionMLP(nn.Module)` with constructor arguments:
  - `data_dim=15` — the residual dimension.
  - `hidden_dim=128` — the MLP hidden width.
  - `t_embed_dim=64` — the sinusoidal embedding dimension.
  - `t_hidden_dim=128` — the TimestepEmbedding output dimension.
  - `n_layers=3` — the number of hidden layers in the main MLP.
  - `use_timestep_embedding=True` — the ablation flag.
- When `use_timestep_embedding=True`:
  - Build a `TimestepEmbedding(t_embed_dim, t_hidden_dim)` and store it.
  - Set the main MLP's input dimension to `data_dim + t_hidden_dim`.
- When `use_timestep_embedding=False`:
  - Do not build a TimestepEmbedding.
  - Set the main MLP's input dimension to just `data_dim`.
- Build the main MLP as `n_layers` hidden layers of `hidden_dim` units
  each, with SiLU activations between them, and a final
  `Linear(hidden_dim, data_dim)` output layer (no activation on the
  output — ε can take any real value).
- The `forward(self, r_t, t)` method:
  - Takes `r_t` of shape `(B, data_dim)` and `t` of shape `(B,)`.
  - When the flag is True, computes `t_emb = self.t_embedding(t)` and
    concatenates with `r_t` along the feature dimension to form the input.
  - When the flag is False, the input is just `r_t`.
  - Returns the MLP output, of shape `(B, data_dim)`.

The forward signature `(r_t, t) → eps_hat` is the contract the
LightningModule in Task 40 will rely on. Both the True and False
configurations satisfy this contract; in the False case, the network
simply ignores `t` entirely.


In [ ]:
# Put your code here for Task 38.


---
## Task 39 — Sanity-test the model, including the t-sensitivity check

Before training anything, two model bugs to rule out. The first is
mundane: shape mismatches at the input or output of the MLP, which will
raise a clear PyTorch error. The second is much more dangerous: a model
that *looks* fine — accepts (r_t, t), produces an output of the right
shape, trains without error — but silently ignores t even when
`use_timestep_embedding=True`. This can happen if the timestep embedding
is implemented but its output is dropped before the concatenation, if the
concatenation axis is wrong, or if the embedding produces all-zero or
all-NaN outputs. None of these failures raise. They only become visible
when the ablation comparison in Task 44 produces "the timestep embedding
doesn't help" — and at that point you cannot tell whether the embedding
is genuinely unhelpful or simply broken.

The defense is the **t-sensitivity check**: hold `r_t` fixed, vary `t`,
and verify that the model's output measurably changes. For
`use_timestep_embedding=True` the change should be non-trivial; for
`use_timestep_embedding=False` the change should be exactly zero (the
network has no path through which t can affect its output).

**Tasks:**
- Instantiate two models:
  - `model_full = DiffusionMLP(use_timestep_embedding=True)`.
  - `model_blind = DiffusionMLP(use_timestep_embedding=False)`.
- For each model:
  - Construct a random batch `r_t = torch.randn(8, 15)` and a random
    `t = torch.randint(0, T, (8,))`.
  - Run `model(r_t, t)` and confirm the output has shape `(8, 15)` with
    no NaNs.
  - Print the total parameter count
    (`sum(p.numel() for p in model.parameters())`). The full model should
    have *more* parameters than the blind model — the difference is exactly
    the parameter count of the TimestepEmbedding module, which the blind
    model does not build.
- **t-sensitivity check (the critical one):**
  - Construct a single fixed input `r_t_fixed = torch.randn(15)` (shape `(15,)`).
  - Construct five timesteps `t_values = torch.tensor([0, T//4, T//2, 3*T//4, T-1])`.
  - Use `einops.repeat(r_t_fixed, 'd -> n d', n=5)` to build a `(5, 15)`
    batch with identical r_t but different t. Do not use `expand` or `tile`.
  - For each model, compute the model output and then the pairwise L2
    distance between rows (a 5 × 5 matrix). For `model_full`, the
    off-diagonal entries should be visibly non-zero. For `model_blind`,
    they should be exactly zero
    (`torch.allclose(output_blind[i], output_blind[j])` should hold for
    any i, j).
  - Display the two 5 × 5 distance matrices as heatmaps side by side. The
    full-model heatmap should show off-diagonal structure; the
    blind-model heatmap should be uniformly zero off-diagonal (a flat
    color map).

If the full-model heatmap is also uniformly zero off-diagonal, your
TimestepEmbedding is broken. Fix it before proceeding to Task 40 — every
downstream task will silently misbehave otherwise.


In [ ]:
# Put your code here for Task 39.


---
## Task 40 — Wrap the model in a LightningModule

The `DiffusionMLP` from Task 38 is a `nn.Module`: it knows how to compute
ε̂ from (r_t, t), but it does not know what to train on, what loss to
minimize, what optimizer to use, or how to use the noise schedule.
PyTorch Lightning's `LightningModule` is the wrapper that adds these
training-specific responsibilities. When you write a Lightning module,
you are answering five questions:

1. *What submodules does the model contain?* (`__init__`)
2. *What constants does it need that are not learnable?* (`register_buffer`)
3. *What does one training step look like?* (`training_step`)
4. *What does one validation step look like?* (`validation_step`)
5. *What optimizer trains it?* (`configure_optimizers`)

For diffusion, the diffusion-specific content sits almost entirely in
question 3 — the training step is where the schedule, the forward
corruption, and the MSE-on-ε loss all converge. The other four questions
have boilerplate-ish answers.

A specific design point: the schedule arrays α and σ from Task 28 of
Week 07 are *constants* (no learnable parameters) but they need to live
on the same device as the model and be saved with the checkpoint. The
PyTorch idiom for that is `register_buffer` — call it in `__init__` with
the precomputed schedule tensors. They will then sit alongside the
model's parameters in the state dict, follow the model to GPU
automatically, and not get updated by the optimizer.

The arrays `alpha_np` and `sigma_np` are already in scope from the setup
cell — same cosine schedule formula as Week 07, recomputed here so this
notebook is self-contained. We pass them as constructor arguments and
register them as buffers, rather than recomputing the cosine formula
inside `__init__`. This keeps the schedule choice visible at the call
site and makes the LightningModule's `training_step` schedule-agnostic:
swap `alpha_np`, `sigma_np` for arrays from a linear schedule and nothing
in the training step changes.

**Tasks:**
- Define a class `DiffusionLightning(pl.LightningModule)`.
- Constructor takes:
  - `model` — an instantiated `DiffusionMLP`.
  - `alpha`, `sigma` — the 1D arrays from the setup cell (each of length T+1).
  - `T` — the number of timesteps (200).
  - `lr=1e-3` — the Adam learning rate.
- In the constructor:
  - Store the model as `self.model`.
  - Convert `alpha` and `sigma` to `torch.float32` tensors and register
    them as buffers (named `'alpha'` and `'sigma'`).
  - Store `T` and `lr` as attributes.
- Implement `training_step(self, batch, batch_idx)`:
  - `batch` is a tensor of clean residuals, shape `(B, 15)`. Call it
    `r_clean`.
  - Sample `t = torch.randint(0, self.T, (B,), device=r_clean.device)`.
  - Draw `eps = torch.randn_like(r_clean)`.
  - Look up `alpha_t = self.alpha[t]` and `sigma_t = self.sigma[t]`,
    each of shape `(B,)`.
  - Use `einops.rearrange(alpha_t, 'b -> b 1')` (and similarly for sigma_t)
    so they broadcast cleanly against `r_clean` of shape `(B, 15)`.
  - Compute `r_t = alpha_t * r_clean + sigma_t * eps`.
  - Compute `eps_hat = self.model(r_t, t)`.
  - Compute `loss = F.mse_loss(eps_hat, eps)`.
  - Log via `self.log('train_loss', loss, prog_bar=True)`.
  - Return `loss`.
- Implement `validation_step(self, batch, batch_idx)`:
  - Same structure as `training_step`, but log as `'val_loss'` and do not
    set `prog_bar=True`.
- Implement `configure_optimizers(self)`:
  - Return `torch.optim.Adam(self.parameters(), lr=self.lr)`.

**Conceptual note worth pausing on:** the `training_step` is fully
*schedule-agnostic*. It looks up `self.alpha[t]` and `self.sigma[t]`
from the buffers and uses them; it does not know whether those values
came from a cosine, linear, or any other schedule formula. Swapping
schedules later (Week 09 onward, if you want to compare schedule
families) will only require changing what you pass to the constructor
— not a single line of `training_step` will change. This is the
separation of concerns we want to preserve as the pipeline grows.


In [ ]:
# Put your code here for Task 40.


---
## Task 41 — Train the unconditional diffusion model

With the dataset, dataloaders, model, and Lightning module in place, the
training loop becomes a few lines of glue: instantiate everything, hand
to a `Trainer`, and call `fit`. This is the payoff of the AI/ML pipeline
abstraction — the wiring you spent Tasks 33–40 building lets the actual
training command be tiny.

We will use **Weights & Biases (WandB)** for experiment logging. WandB
gives you live loss curves, hyperparameter tracking, and a comparison
view across runs that will be especially useful when the ablation in
Task 44 produces a second run to compare against the first. If you have
not used WandB before, the first run will prompt you to log in (free
account, follow the URL it gives you).

**Tasks:**
- Instantiate `model_full = DiffusionMLP(use_timestep_embedding=True)`.
- Instantiate
  `lightning_full = DiffusionLightning(model_full, alpha=alpha_np, sigma=sigma_np, T=T)`.
- Create a `WandbLogger` with `project='butterflai-wk08'` and a clear run
  name like `'unconditional_full'`.
- Create a `pl.Trainer`:
  - `max_epochs=200` (15-dim data is small; epochs are fast).
  - `logger=wandb_logger`.
  - `accelerator='auto'`, `devices='auto'`.
  - `log_every_n_steps=10`.
- Call `trainer.fit(lightning_full, train_loader, val_loader)`.
- After training completes, save the model checkpoint to a known path
  (e.g. `'./ckpt_full.ckpt'`) using `trainer.save_checkpoint(...)`.
- Plot the training and validation loss curves from the logged history.
  Both should decrease, with the val loss eventually plateauing or
  rising slowly if the model overfits.

**Important callout — what loss does and does not tell you:** a decreasing
training loss is necessary but **not sufficient** for a good diffusion
model. The training loss is the MSE between the network's predicted ε
and the true ε; it measures how well the network estimates the noise
that was added during corruption. It does *not* directly measure whether
the samples drawn from the trained model look like the training data. A
model can have a low MSE-on-ε and still produce samples that are
visually wrong — for example, if the network has memorized the marginal
mean of ε but not the per-r_t structure. We will evaluate sample quality
directly in Task 43, after we have built a sampler in Task 42.


In [ ]:
# Put your code here for Task 41.


---
## Task 42 — Implement a DDIM-style sampler

Sampling is the *inference* counterpart of training. Training corrupts
clean residuals to noise: we take a clean r, pick a random t, and produce
r_t. Sampling reverses that path: we start from pure noise r_T ~ N(0, I)
and step backward through the schedule until we arrive at a clean r_0 —
a generated residual.

There is an asymmetry between training and sampling worth pausing on.
Training is *parallel* over t: every sample in a batch gets an independent
random t, all denoised in a single forward pass through the network.
Sampling is *sequential*: you must walk the schedule from t = T-1 down to
t = 0, calling the network at every step. With T = 200 timesteps, sampling
one batch of residuals takes 200 sequential forward passes; training one
batch takes one. This is why training scales pleasantly with batch size
but sampling does not.

The reverse step we implement is **deterministic DDIM** (Song, Meng,
Ermon 2020). DDIM is mathematically related to the more famous DDPM
(Ho, Jain, Abbeel 2020), but DDIM removes the per-step random noise and
makes the sampling process deterministic given the starting r_T. The
advantages: simpler code, easier debugging, identical sample quality with
many fewer steps if you want to skip timesteps. The DDIM step using the
α_t / σ_t parameterization from Week 07 is:

```
ε̂        = model(r_t, t)
r̂_0      = (r_t - σ_t · ε̂) / α_t              # estimate the clean residual
r_{t-1}  = α_{t-1} · r̂_0 + σ_{t-1} · ε̂        # step to t-1 along the same direction
```

Read the second line carefully: we use the *same* ε̂ from the model both
to estimate r̂_0 and to step to t-1. That is the essence of the
deterministic DDIM step.

**Tasks:**
- Implement a function
  `sample(lightning_module, batch_size, data_dim, device)` that returns
  a batch of generated residuals of shape `(batch_size, data_dim)`. The
  function should pull `alpha`, `sigma`, and `T` from the
  `lightning_module`'s buffers and attributes.
- The function should:
  - Set `lightning_module.eval()` and wrap the body in `torch.no_grad()`.
  - Initialize `r_t = torch.randn(batch_size, data_dim, device=device)`.
  - Loop `t` from `T - 1` down to `0`:
    - Construct
      `t_batch = torch.full((batch_size,), t, dtype=torch.long, device=device)`.
    - Compute `eps_hat = lightning_module.model(r_t, t_batch)`.
    - Look up `alpha_t = lightning_module.alpha[t]`,
      `sigma_t = lightning_module.sigma[t]` (scalars).
    - Compute `r_0_hat = (r_t - sigma_t * eps_hat) / alpha_t`.
    - If `t > 0`:
      `r_t = lightning_module.alpha[t-1] * r_0_hat + lightning_module.sigma[t-1] * eps_hat`.
    - If `t == 0`: `r_t = r_0_hat` (the final estimate is the clean sample).
  - Return `r_t`.

**Sanity check:** call `sample(lightning_full, batch_size=8, data_dim=15, device=device)`
and verify the output has shape `(8, 15)` with no NaNs. Plot the eight
samples as bar charts on a 2 × 4 grid using `BIN_CENTERS`. They should
look like residuals (zero-mean-ish, structure in middle bins, small at
edges) — but at this point all you can confirm is "they don't look
obviously broken." The quantitative comparison against the training
distribution is Task 43.

**Note on alternatives:** The stochastic DDPM step adds a fresh ε draw at
each timestep, producing different samples from the same starting r_T.
DDPM is the more conventional choice in the literature; DDIM is what we
use here for pedagogical simplicity. If you finish early and are curious,
add a `stochastic=True` flag and implement the DDPM step
`r_{t-1} = α_{t-1} · r̂_0 + sqrt(σ_{t-1}^2 - eta^2) · ε̂ + eta · z` where
`z ~ N(0, I)` and `eta` is the step's noise level (commonly η = σ_{t-1}
for full DDPM, η = 0 for DDIM).


In [ ]:
# Put your code here for Task 42.


---
## Task 43 — Verify the sampled distribution matches the training distribution

The training loss tells you the network is learning *something*; it does
not tell you the network has learned the *right* thing. The direct
empirical question is: do samples from the trained model look like real
training residuals? At the level of distributional statistics, if the
model has learned the marginal residual distribution well, its samples
should match the training residuals on bin-wise mean, bin-wise standard
deviation, and bin-bin covariance structure.

This task asks you to compute and visualize that comparison. The
diagnostics are the same three you used in Task 31 of Week 07 to verify
the forward process — bin-wise mean, bin-wise standard deviation, and
covariance matrix — applied here to two distributions: the empirical
training residuals and the model-generated residuals.

**Tasks:**
- Sample 1000 residuals from the trained full model:
  `samples_full = sample(lightning_full, batch_size=1000, data_dim=15, device=device).cpu().numpy()`.
- Collect 1000 random training residuals into a numpy array of shape
  `(1000, 15)` (sample with replacement from `train_dataset` if needed).
- Compute the **bin-wise mean** for both, plot side-by-side as bar charts
  using `BIN_CENTERS`. They should agree within ~`1/sqrt(1000) ≈ 3%` of
  typical residual scale.
- Compute the **bin-wise standard deviation** for both, plot side-by-side.
  They should agree similarly.
- Compute the **bin-bin covariance matrix** (15 × 15) for both, plot
  side-by-side as heatmaps with the same color scale. The model's
  covariance should reproduce the diagonal (per-bin variance) and the
  dominant off-diagonal structure (correlations between neighboring bins,
  anti-correlations between distant bins) of the training data.
- **Visual overlay:** plot 20 random training residuals and 20 random
  model samples on the same axes (different colors). They should be
  visually indistinguishable — same overall envelope, same kind of
  structure in the middle bins, same near-zero behavior at edges. If the
  model samples are visibly smoother or more uniform than the training
  residuals, the model has under-learned. If they are visibly noisier or
  spikier, the model has under-trained or has a bug.
- Compute and report a single scalar summary: the **bin-wise MSE** between
  the sampled bin-wise mean and the training bin-wise mean. We will reuse
  this number in Task 44 to compare against the ablation.

**Caveat to flag:** because this week's model is *unconditional*, it is
matching the *marginal* training distribution — averaged across all cycles,
phases, and amplitudes. It is not learning to generate residuals
appropriate for any specific window. That is fine for this week; it is
exactly what the unconditional model is supposed to do. The check
"do generated residuals match the training-set marginals" is the right
test for the unconditional model.

**Optional ad-hoc inspection:** if you want to see what a sampled residual
"adds" to a classical prediction, pick any window from `windows_df`, look
up its `(amplitude, tau_center)`, evaluate `classical.density(A, tau, BIN_CENTERS)`
to get the classical density on the bin grid, integrate it to get the
binned classical histogram, and overlay it with `binned_classical + r_sample`
for a few sampled residuals. The combined predictions will *not*
systematically improve over the classical alone for this specific window —
the residual is a draw from the marginal, not targeted at this window's
amplitude or phase. Conditioning on (amplitude, mu_universal) in Week 09
is what makes the residual targeted; the corresponding `compute_global_nll`
value-add metric arrives there.


In [ ]:
# Put your code here for Task 43.


---
## Task 44 — Run the no-timestep-embedding ablation

The whole reason for the ablation flag in Task 38 is the question this
task answers: does the timestep embedding earn its keep? We trained one
model with `use_timestep_embedding=True`. Now we train a second, identical
in every other respect, with `use_timestep_embedding=False`. The
**ablation comparison** is the two models' performance on the same metrics
from Task 43. If the t-blind model does almost as well, the timestep
embedding is not contributing much (or is broken — Task 39 should have
ruled the latter out). If the t-blind model does substantially worse,
the timestep embedding earns its keep.

There is a useful prediction to commit to before running this task. The
t-blind model has only one possible strategy: it must produce a single
ε estimate that is averaged over all noise levels — a kind of "average
denoiser" that does not adapt to the noise level it is currently looking
at. At low t, where the input is mostly clean, this average denoiser will
add too much noise to the estimate; at high t, where the input is mostly
noise, it will subtract too little. The net effect on samples is
typically that the marginal mean and variance roughly match (because the
overall scale is correct on average) but the covariance structure
collapses — the model loses the per-bin correlations that distinguish
training residuals from white noise.

Predict before you run. Then run.

**Tasks:**
- Instantiate `model_blind = DiffusionMLP(use_timestep_embedding=False)`.
- Instantiate
  `lightning_blind = DiffusionLightning(model_blind, alpha=alpha_np, sigma=sigma_np, T=T)`.
- Train it with the same Trainer configuration as Task 41, but with a
  distinct WandB run name like `'unconditional_blind'`. Use the same
  `max_epochs`, the same DataLoaders, the same learning rate. Save the
  checkpoint to `'./ckpt_blind.ckpt'`.
- Sample 1000 residuals from the trained blind model.
- Re-run all three diagnostics from Task 43 (bin-wise mean, std,
  covariance heatmap) for the blind model.
- Build a comparison table with three rows (training data, full model,
  blind model) and four columns (final train_loss, final val_loss,
  bin-wise mean MSE vs. training, qualitative covariance match yes/no).
- Plot the two covariance heatmaps (full and blind) side by side. The
  qualitative comparison is the headline result: does the blind model's
  covariance match the training data's, or has it collapsed to something
  more diagonal?

**How to read the result:**

- *If the full model clearly beats the blind model* on covariance match:
  the timestep embedding earns its keep. The architectural complexity is
  doing real work, and Week 09's conditioning machinery should be built
  on top of the t-aware base.
- *If the two models are nearly tied:* the timestep embedding is not
  contributing much on this dataset. Possible reasons: 200 timesteps is
  more than this 15-dimensional residual problem needs (a network can
  approximately memorize an "average denoiser" when the noise levels are
  not too varied); the cosine schedule keeps the signal alive across a
  large fraction of t (so the average denoiser is not far from any
  specific denoiser); or the data does not have enough structure across
  noise levels for t-awareness to matter. None of these is a bug — they
  are real findings about this problem at this scale.
- *If the blind model outperforms the full model:* a real bug somewhere,
  most likely in the TimestepEmbedding or in the concatenation. Re-run
  Task 39's t-sensitivity check to localize.

Either of the first two outcomes is publishable insight about your
specific problem. The result is not predetermined — that is what makes
this an ablation rather than a demonstration.


In [ ]:
# Put your code here for Task 44.


---
## Where Week 08 leaves us, and what Week 09 will need

By the end of Task 44 you have built every piece of the AI/ML pipeline a
PyTorch + Lightning project needs, instantiated for the unconditional
diffusion model:

- A **PyTorch Dataset** that wraps the Week 07 residual table loaded
  from `diffusion_windows.parquet` (Task 33).
- A set of **DataLoaders** for train / val / test using the parquet's
  `split` column with appropriate shuffling (Task 35).
- A **sinusoidal timestep embedding** that turns integer t into a dense
  vector (Task 36) and a **TimestepEmbedding module** that learns to
  project that vector into a useful representation (Task 37).
- A **DiffusionMLP** that takes (r_t, t) and predicts ε̂, with a flag for
  the architectural ablation (Task 38), sanity-tested for shape and
  t-sensitivity (Task 39).
- A **LightningModule** that stitches the model and the Week 07 schedule
  into a training loop (Task 40), trained end-to-end with WandB logging
  (Task 41).
- A **DDIM sampler** that runs the trained model in reverse to generate
  new residuals (Task 42).
- Two empirical results: the **distributional verification** that the
  trained model's samples match the training distribution (Task 43) and
  the **timestep-embedding ablation** that measures whether the
  diffusion-specific architecture earns its keep (Task 44).

What you have **not** yet built is conditioning: every sample drawn from
the trained model is from the *marginal* residual distribution, not from
a distribution targeted at a specific window. That is the Week 09 lift.

**What changes in Week 09.** The Dataset returns
`(r, amplitude, mu_universal)` tuples instead of just `r` (the parquet
already has those columns). The DiffusionMLP gains a second concatenation
input — the conditioning vector — alongside the timestep embedding. The
LightningModule's `training_step` passes the conditioning through. The
sampler accepts a per-sample conditioning vector and threads it through
each reverse step. None of the diffusion-specific machinery (the schedule,
the forward equation, the ε-prediction objective, the timestep embedding
itself) changes. The conditioning is an *extension* of the architecture
you have already built, not a replacement for it.

**What stays.** Every line of `training_step`. Every line of the sampler
loop. The LightningModule structure. The schedule buffers. The full
ablation infrastructure. Week 09 is a small refactor on a working
pipeline, not a rebuild.

**What we expect to see in Week 09.** With conditioning, the combined
classical + diffusion model (the official `ButterflAIModel.density(A, tau, ·)`
loaded as `classical` in this notebook's setup, plus a sampled residual
targeted at that window's amplitude and mu_universal) should outperform
the classical model alone on `compute_global_nll` for held-out test
cycles. That is the value-add metric the program has been pointing at
since Week 03. Whether the margin is large or small is itself a real
empirical question, and it is the question Week 09 finally lets you
answer.
